In [1]:
from pathlib import Path
import zipfile
import shutil
import yaml


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

CARPETA_DATA = Path(
    r"C:\MachinL\ProyectoFinal"
)

IMAGENES_ORIGINALES = (
    CARPETA_DATA /
    "Orange_segmentation_manual"
)

EXPORTACIONES_CVAT = (
    CARPETA_DATA /
    "CVAT_exports"
)

DATASET_YOLO = (
    CARPETA_DATA /
    "orange_segmentation_yolo"
)

CARPETA_TEMPORAL = (
    CARPETA_DATA /
    "_cvat_extraido_temporal"
)

EXTENSIONES_IMAGEN = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".tif", ".tiff", ".webp"
}

ARCHIVOS_ZIP = {
    "train": EXPORTACIONES_CVAT / "Orange_Train_YOLO.zip",
    "val": EXPORTACIONES_CVAT / "Orange_Val_YOLO.zip",
    "test": EXPORTACIONES_CVAT / "Orange_Test_YOLO.zip"
}

CARPETAS_ORIGINALES = {
    "train": IMAGENES_ORIGINALES / "Train",
    "val": IMAGENES_ORIGINALES / "Val",
    "test": IMAGENES_ORIGINALES / "Test"
}

In [2]:
# ============================================================
# 2. VALIDAR RUTAS
# ============================================================

if not IMAGENES_ORIGINALES.exists():
    raise FileNotFoundError(
        f"No se encontró:\n{IMAGENES_ORIGINALES}"
    )

if not EXPORTACIONES_CVAT.exists():
    raise FileNotFoundError(
        f"No se encontró:\n{EXPORTACIONES_CVAT}"
    )

for split, ruta_zip in ARCHIVOS_ZIP.items():
    if not ruta_zip.exists():
        raise FileNotFoundError(
            f"No se encontró el ZIP de {split}:\n"
            f"{ruta_zip}"
        )

for split, carpeta in CARPETAS_ORIGINALES.items():
    if not carpeta.exists():
        raise FileNotFoundError(
            f"No se encontró la carpeta de {split}:\n"
            f"{carpeta}"
        )

print("Todas las rutas fueron encontradas.")

Todas las rutas fueron encontradas.


In [3]:
def obtener_imagenes(carpeta):
    return sorted([
        archivo
        for archivo in carpeta.rglob("*")
        if (
            archivo.is_file()
            and archivo.suffix.lower()
            in EXTENSIONES_IMAGEN
        )
    ])


def es_archivo_anotacion_yolo(ruta_txt):
    """
    Identifica archivos de polígonos YOLO y excluye
    archivos como train.txt, obj.names o listas internas.
    """
    try:
        contenido = ruta_txt.read_text(
            encoding="utf-8"
        ).strip()

        if not contenido:
            return False

        primera_linea = contenido.splitlines()[0]
        valores = primera_linea.split()

        # Segmentación YOLO:
        # clase x1 y1 x2 y2 x3 y3 ...
        if len(valores) < 7:
            return False

        int(valores[0])

        coordenadas = [
            float(valor)
            for valor in valores[1:]
        ]

        return all(
            0.0 <= valor <= 1.0
            for valor in coordenadas
        )

    except Exception:
        return False

In [4]:
if DATASET_YOLO.exists():
    shutil.rmtree(DATASET_YOLO)

if CARPETA_TEMPORAL.exists():
    shutil.rmtree(CARPETA_TEMPORAL)

for split in ["train", "val", "test"]:
    (DATASET_YOLO / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )

    (DATASET_YOLO / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )

CARPETA_TEMPORAL.mkdir(
    parents=True,
    exist_ok=True
)

print("Carpetas creadas.")

Carpetas creadas.


In [5]:
resumen = {}

for split in ["train", "val", "test"]:

    print(f"\nProcesando: {split}")

    carpeta_extraida = (
        CARPETA_TEMPORAL /
        split
    )

    carpeta_extraida.mkdir(
        parents=True,
        exist_ok=True
    )

    # Extraer ZIP de CVAT
    with zipfile.ZipFile(
        ARCHIVOS_ZIP[split],
        "r"
    ) as archivo_zip:

        archivo_zip.extractall(
            carpeta_extraida
        )

    # Crear índice de imágenes originales por nombre base
    imagenes = obtener_imagenes(
        CARPETAS_ORIGINALES[split]
    )

    indice_imagenes = {}

    for imagen in imagenes:
        indice_imagenes.setdefault(
            imagen.stem,
            []
        ).append(imagen)

    # Buscar archivos de polígonos
    archivos_txt = [
        archivo
        for archivo in carpeta_extraida.rglob("*.txt")
        if es_archivo_anotacion_yolo(archivo)
    ]

    copiadas = 0
    no_encontradas = []
    duplicadas = []

    for etiqueta in archivos_txt:

        nombre_base = etiqueta.stem

        coincidencias = indice_imagenes.get(
            nombre_base,
            []
        )

        if len(coincidencias) == 0:
            no_encontradas.append(
                etiqueta
            )
            continue

        if len(coincidencias) > 1:
            duplicadas.append(
                nombre_base
            )
            continue

        imagen_origen = coincidencias[0]

        imagen_destino = (
            DATASET_YOLO /
            "images" /
            split /
            imagen_origen.name
        )

        etiqueta_destino = (
            DATASET_YOLO /
            "labels" /
            split /
            f"{imagen_origen.stem}.txt"
        )

        shutil.copy2(
            imagen_origen,
            imagen_destino
        )

        shutil.copy2(
            etiqueta,
            etiqueta_destino
        )

        copiadas += 1

    resumen[split] = {
        "imagenes_originales": len(imagenes),
        "anotaciones_encontradas": len(archivos_txt),
        "pares_copiados": copiadas,
        "sin_imagen": len(no_encontradas),
        "nombres_duplicados": len(duplicadas)
    }

    print(
        f"Imágenes originales: {len(imagenes)}"
    )

    print(
        f"Anotaciones encontradas: {len(archivos_txt)}"
    )

    print(
        f"Pares copiados: {copiadas}"
    )

    if no_encontradas:
        print(
            f"Anotaciones sin imagen: "
            f"{len(no_encontradas)}"
        )

        for archivo in no_encontradas[:5]:
            print("  ", archivo.name)

    if duplicadas:
        print(
            f"Nombres duplicados: "
            f"{len(duplicadas)}"
        )


Procesando: train
Imágenes originales: 100
Anotaciones encontradas: 100
Pares copiados: 100

Procesando: val
Imágenes originales: 25
Anotaciones encontradas: 25
Pares copiados: 25

Procesando: test
Imágenes originales: 25
Anotaciones encontradas: 25
Pares copiados: 25


In [6]:
contenido_yaml = {
    "path": str(
        DATASET_YOLO.resolve()
    ),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "orange"
    }
}

ruta_yaml = (
    DATASET_YOLO /
    "data.yaml"
)

with open(
    ruta_yaml,
    "w",
    encoding="utf-8"
) as archivo:

    yaml.safe_dump(
        contenido_yaml,
        archivo,
        sort_keys=False,
        allow_unicode=True
    )

print("\nArchivo creado:")
print(ruta_yaml)


Archivo creado:
C:\MachinL\ProyectoFinal\orange_segmentation_yolo\data.yaml


In [7]:
print("\n===================================")
print("RESUMEN FINAL")
print("===================================")

for split, datos in resumen.items():
    print(f"\n{split.upper()}")

    for nombre, valor in datos.items():
        print(
            f"  {nombre}: {valor}"
        )


RESUMEN FINAL

TRAIN
  imagenes_originales: 100
  anotaciones_encontradas: 100
  pares_copiados: 100
  sin_imagen: 0
  nombres_duplicados: 0

VAL
  imagenes_originales: 25
  anotaciones_encontradas: 25
  pares_copiados: 25
  sin_imagen: 0
  nombres_duplicados: 0

TEST
  imagenes_originales: 25
  anotaciones_encontradas: 25
  pares_copiados: 25
  sin_imagen: 0
  nombres_duplicados: 0
